# 02 - Exploratory Data Analysis: Food.com Dataset

**Project:** GroundedNutriRec  
**Student:** Student 1 - Data + Sequential Recommendation Lead  
**Scope:** Comprehensive EDA of RAW_recipes.csv and RAW_interactions.csv  
**Goal:** Understand data distributions, identify outliers, compute sparsity, and characterize user/recipe patterns.

## 1. Environment Setup

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from SRC.data_preprocessing import (
    load_raw_recipes,
    load_raw_interactions,
    remove_time_outliers,
    derive_implicit_feedback,
    NUTRITION_COLUMN_NAMES,
    MINUTES_OUTLIER_THRESHOLD,
)

sns.set_theme(style='whitegrid', font_scale=1.1)
FIGURE_DIR = project_root / 'RESULTS' / 'FIGURES'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project Root: {project_root}')

Project Root: C:\Users\Kush Shah\OneDrive\Desktop\Internship


## 2. Load Data

In [2]:
recipes_df = load_raw_recipes(parse_lists=True, parse_nutrition=True)
interactions_df = load_raw_interactions()

print(f'Recipes shape:      {recipes_df.shape}')
print(f'Interactions shape:  {interactions_df.shape}')

Recipes shape:      (231637, 19)
Interactions shape:  (1132367, 5)


## 3. Distribution of Cooking Time (minutes)

Many recipes have extreme preparation times (some report 1,000,000+ minutes).  
We visualize both the raw distribution and the cleaned distribution  
after removing outliers above 1440 minutes (24 hours).

In [3]:
print('Cooking time statistics (raw):')
print(recipes_df['minutes'].describe())
print(f'\nRecipes above {MINUTES_OUTLIER_THRESHOLD} min: {(recipes_df["minutes"] > MINUTES_OUTLIER_THRESHOLD).sum():,}')

Cooking time statistics (raw):
count    2.316370e+05
mean     9.398546e+03
std      4.461963e+06
min      0.000000e+00
25%      2.000000e+01
50%      4.000000e+01
75%      6.500000e+01
max      2.147484e+09
Name: minutes, dtype: float64

Recipes above 1440 min: 2,000


In [4]:
recipes_clean = remove_time_outliers(recipes_df)
print(f'Shape after removing time outliers: {recipes_clean.shape}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(recipes_df['minutes'].clip(upper=500), bins=100, color='#3498db', edgecolor='white')
axes[0].set_title('Cooking Time Distribution (Raw, clipped at 500 min)')
axes[0].set_xlabel('Minutes')
axes[0].set_ylabel('Frequency')

axes[1].hist(recipes_clean['minutes'], bins=100, color='#2ecc71', edgecolor='white')
axes[1].set_title(f'Cooking Time Distribution (Cleaned, <= {MINUTES_OUTLIER_THRESHOLD} min)')
axes[1].set_xlabel('Minutes')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig(FIGURE_DIR / 'cooking_time_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

Shape after removing time outliers: (229637, 19)


C:\Users\Kush Shah\AppData\Local\Temp\ipykernel_16480\1802744896.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Distribution of Number of Ingredients

In [5]:
print('n_ingredients statistics:')
print(recipes_clean['n_ingredients'].describe())

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(recipes_clean['n_ingredients'], bins=range(0, 40), color='#e74c3c', edgecolor='white', alpha=0.85)
ax.set_title('Distribution of Number of Ingredients per Recipe')
ax.set_xlabel('Number of Ingredients')
ax.set_ylabel('Frequency')
ax.axvline(recipes_clean['n_ingredients'].median(), color='black', linestyle='--',
           label=f'Median = {recipes_clean["n_ingredients"].median():.0f}')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'n_ingredients_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

n_ingredients statistics:
count    229637.000000
mean          9.051242
std           3.728369
min           1.000000
25%           6.000000
50%           9.000000
75%          11.000000
max          43.000000
Name: n_ingredients, dtype: float64


C:\Users\Kush Shah\AppData\Local\Temp\ipykernel_16480\1419670915.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Distribution of Number of Steps

In [6]:
print('n_steps statistics:')
print(recipes_clean['n_steps'].describe())

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(recipes_clean['n_steps'], bins=range(0, 50), color='#9b59b6', edgecolor='white', alpha=0.85)
ax.set_title('Distribution of Number of Steps per Recipe')
ax.set_xlabel('Number of Steps')
ax.set_ylabel('Frequency')
ax.axvline(recipes_clean['n_steps'].median(), color='black', linestyle='--',
           label=f'Median = {recipes_clean["n_steps"].median():.0f}')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'n_steps_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

n_steps statistics:
count    229637.000000
mean          9.739467
std           5.949927
min           0.000000
25%           6.000000
50%           9.000000
75%          12.000000
max         145.000000
Name: n_steps, dtype: float64


C:\Users\Kush Shah\AppData\Local\Temp\ipykernel_16480\2021596331.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Nutrition Analysis (7-element PDV Vector)

The `nutrition` column in RAW_recipes.csv encodes 7 values:  
`[calories, total_fat_pdv, sugar_pdv, sodium_pdv, protein_pdv, saturated_fat_pdv, carbohydrates_pdv]`  

These have been decomposed into separate columns by `load_raw_recipes()`.

In [7]:
# Summary statistics for all 7 nutrition columns
recipes_clean[NUTRITION_COLUMN_NAMES].describe()

,calories,total_fat_pdv,sugar_pdv,sodium_pdv,protein_pdv,saturated_fat_pdv,carbohydrates_pdv
count,229637.000000,229637.000000,229637.000000,229637.000000,229637.000000,229637.000000,229637.00000
mean,471.281117,35.956070,82.895587,29.644421,34.600539,45.463048,15.44114
std,1184.944660,77.267852,797.178170,126.722517,58.278292,97.078292,81.59926
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
25%,174.300000,8.000000,9.000000,5.000000,7.000000,7.000000,4.00000
50%,313.000000,20.000000,25.000000,14.000000,18.000000,23.000000,9.00000
75%,518.100000,41.000000,67.000000,33.000000,51.000000,52.000000,16.00000
max,434360.200000,17183.000000,362729.000000,29338.000000,6552.000000,10395.000000,36098.00000


In [8]:
# Box plots for nutrition columns (clipping extreme outliers for visibility)
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

colors = ['#e74c3c', '#3498db', '#e67e22', '#2ecc71', '#9b59b6', '#f39c12', '#1abc9c']

for idx, col_name in enumerate(NUTRITION_COLUMN_NAMES):
    data_clipped = recipes_clean[col_name].clip(upper=recipes_clean[col_name].quantile(0.99))
    axes[idx].boxplot(data_clipped.dropna(), vert=True, patch_artist=True,
                      boxprops=dict(facecolor=colors[idx], alpha=0.7))
    axes[idx].set_title(col_name)
    axes[idx].set_ylabel('Value')

axes[7].axis('off')
plt.suptitle('Nutrition PDV Distributions (clipped at 99th percentile)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'nutrition_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\Kush Shah\AppData\Local\Temp\ipykernel_16480\2518206061.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# Correlation matrix between nutrition columns
fig, ax = plt.subplots(figsize=(9, 7))
nutrition_corr = recipes_clean[NUTRITION_COLUMN_NAMES].corr()
sns.heatmap(nutrition_corr, annot=True, fmt='.2f', cmap='RdYlBu_r', center=0,
            square=True, ax=ax, linewidths=0.5)
ax.set_title('Nutrition Column Correlation Matrix')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'nutrition_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\Kush Shah\AppData\Local\Temp\ipykernel_16480\1172802987.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Rating Distribution (RAW_interactions.csv)

Ratings range from 0 to 5.  
- Rating = 0 indicates no explicit rating was given (user left a review without rating).  
- For implicit feedback: rating >= 4 -> liked=1, 0 < rating < 4 -> liked=0, rating=0 -> NaN.

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw rating distribution
rating_counts = interactions_df['rating'].value_counts().sort_index()
axes[0].bar(rating_counts.index, rating_counts.values, color='#3498db', edgecolor='white')
axes[0].set_title('Rating Distribution (Raw)')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')
for idx_r, val in enumerate(rating_counts.values):
    axes[0].text(rating_counts.index[idx_r], val + 5000, f'{val:,}', ha='center', fontsize=9)

# Implicit feedback distribution
interactions_implicit = derive_implicit_feedback(interactions_df)
liked_counts = interactions_implicit['liked'].value_counts(dropna=False).sort_index()
labels = ['NaN (rating=0)', 'Disliked (0)', 'Liked (1)']
colors_bar = ['#95a5a6', '#e74c3c', '#2ecc71']
axes[1].bar(range(len(liked_counts)), liked_counts.values, color=colors_bar[:len(liked_counts)], edgecolor='white')
axes[1].set_xticks(range(len(liked_counts)))
axes[1].set_xticklabels(labels[:len(liked_counts)])
axes[1].set_title('Implicit Feedback Distribution')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig(FIGURE_DIR / 'rating_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\Kush Shah\AppData\Local\Temp\ipykernel_16480\3174728125.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. User Activity Distribution

Number of interactions per user.  
Highly skewed: most users have very few interactions, a small number are power users.

In [11]:
user_activity = interactions_df.groupby('user_id').size().reset_index(name='interaction_count')
print('User activity statistics:')
print(user_activity['interaction_count'].describe())
print(f'\nUsers with only 1 interaction: {(user_activity["interaction_count"] == 1).sum():,}')
print(f'Users with >= 10 interactions:  {(user_activity["interaction_count"] >= 10).sum():,}')
print(f'Users with >= 50 interactions:  {(user_activity["interaction_count"] >= 50).sum():,}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(user_activity['interaction_count'].clip(upper=100), bins=100, color='#e67e22', edgecolor='white')
ax.set_title('User Activity Distribution (clipped at 100 interactions)')
ax.set_xlabel('Number of Interactions per User')
ax.set_ylabel('Number of Users')
ax.set_yscale('log')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'user_activity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

User activity statistics:
count    226570.000000
mean          4.997868
std          49.663111
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max        7671.000000
Name: interaction_count, dtype: float64

Users with only 1 interaction: 166,256
Users with >= 10 interactions:  12,486
Users with >= 50 interactions:  2,755


C:\Users\Kush Shah\AppData\Local\Temp\ipykernel_16480\3829089874.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Recipe Popularity Distribution

Number of interactions per recipe.  
Also highly skewed: a few popular recipes receive hundreds of reviews.

In [12]:
recipe_popularity = interactions_df.groupby('recipe_id').size().reset_index(name='interaction_count')
print('Recipe popularity statistics:')
print(recipe_popularity['interaction_count'].describe())

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(recipe_popularity['interaction_count'].clip(upper=50), bins=50, color='#1abc9c', edgecolor='white')
ax.set_title('Recipe Popularity Distribution (clipped at 50 interactions)')
ax.set_xlabel('Number of Interactions per Recipe')
ax.set_ylabel('Number of Recipes')
ax.set_yscale('log')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'recipe_popularity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

Recipe popularity statistics:
count    231637.000000
mean          4.888541
std          17.532481
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max        1613.000000
Name: interaction_count, dtype: float64


C:\Users\Kush Shah\AppData\Local\Temp\ipykernel_16480\1931352530.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Interaction Matrix Sparsity

Sparsity = 1 - (total_interactions / (unique_users x unique_recipes))  
This metric quantifies how much of the user-item matrix is empty.

In [13]:
unique_users = interactions_df['user_id'].nunique()
unique_recipes = interactions_df['recipe_id'].nunique()
total_interactions = len(interactions_df)
matrix_size = unique_users * unique_recipes
sparsity = 1 - (total_interactions / matrix_size)

print(f'Unique users:        {unique_users:,}')
print(f'Unique recipes:      {unique_recipes:,}')
print(f'Total interactions:   {total_interactions:,}')
print(f'Matrix size:         {matrix_size:,}')
print(f'Observed entries:    {total_interactions:,}')
print(f'Sparsity:            {sparsity:.8f} ({sparsity*100:.5f}%)')

Unique users:        226,570
Unique recipes:      231,637
Total interactions:   1,132,367
Matrix size:         52,481,995,090
Observed entries:    1,132,367
Sparsity:            0.99997842 (99.99784%)


## 11. Temporal Patterns

Submission dates (recipes) and interaction dates span 18+ years.  
Understanding temporal density is critical for sequential recommendation in Week 6.

In [14]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Recipe submissions per year
recipes_clean['submitted_year'] = recipes_clean['submitted'].dt.year
submissions_per_year = recipes_clean['submitted_year'].value_counts().sort_index()
axes[0].bar(submissions_per_year.index, submissions_per_year.values, color='#3498db', edgecolor='white')
axes[0].set_title('Recipe Submissions per Year')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Number of Recipes')

# Interactions per year
interactions_df['date_year'] = interactions_df['date'].dt.year
interactions_per_year = interactions_df['date_year'].value_counts().sort_index()
axes[1].bar(interactions_per_year.index, interactions_per_year.values, color='#e74c3c', edgecolor='white')
axes[1].set_title('User Interactions per Year')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Number of Interactions')

plt.tight_layout()
plt.savefig(FIGURE_DIR / 'temporal_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\Kush Shah\AppData\Local\Temp\ipykernel_16480\4092840843.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 12. Tag Frequency Analysis

Each recipe has a list of tags (e.g., 'vegetarian', '30-minutes-or-less', 'healthy').  
Tag frequency reveals the dominant recipe categories in the corpus.

In [15]:
from collections import Counter

all_tags = [tag for tag_list in recipes_clean['tags'].dropna() for tag in tag_list]
tag_counter = Counter(all_tags)
top_tags = tag_counter.most_common(25)

print(f'Total unique tags: {len(tag_counter):,}')
print(f'\nTop 25 tags:')
for tag, count in top_tags:
    print(f'  {tag}: {count:,}')

Total unique tags: 552

Top 25 tags:
  preparation: 228,551
  time-to-make: 223,332
  course: 216,292
  main-ingredient: 168,998
  dietary: 163,694
  easy: 125,050
  occasion: 113,026
  cuisine: 90,248
  low-in-something: 85,038
  main-dish: 71,216
  60-minutes-or-less: 69,989
  equipment: 69,770
  number-of-servings: 58,411
  meat: 55,446
  30-minutes-or-less: 55,076
  vegetables: 53,501
  taste-mood: 51,702
  4-hours-or-less: 49,493
  north-american: 48,047
  3-steps-or-less: 44,728
  15-minutes-or-less: 43,934
  low-sodium: 42,965
  desserts: 42,890
  low-carb: 41,813
  healthy: 39,960


In [16]:
fig, ax = plt.subplots(figsize=(12, 7))
tag_names = [t[0] for t in top_tags]
tag_counts = [t[1] for t in top_tags]
ax.barh(tag_names[::-1], tag_counts[::-1], color='#9b59b6', edgecolor='white')
ax.set_title('Top 25 Most Frequent Tags')
ax.set_xlabel('Frequency')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'top_tags.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\Kush Shah\AppData\Local\Temp\ipykernel_16480\2774049849.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 13. Ingredient Frequency Analysis

In [17]:
all_ingredients = [ing for ing_list in recipes_clean['ingredients'].dropna() for ing in ing_list]
ingredient_counter = Counter(all_ingredients)
top_ingredients = ingredient_counter.most_common(25)

print(f'Total unique ingredients: {len(ingredient_counter):,}')
print(f'\nTop 25 ingredients:')
for ing, count in top_ingredients:
    print(f'  {ing}: {count:,}')

fig, ax = plt.subplots(figsize=(12, 7))
ing_names = [i[0] for i in top_ingredients]
ing_counts = [i[1] for i in top_ingredients]
ax.barh(ing_names[::-1], ing_counts[::-1], color='#e67e22', edgecolor='white')
ax.set_title('Top 25 Most Frequent Ingredients')
ax.set_xlabel('Frequency')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'top_ingredients.png', dpi=150, bbox_inches='tight')
plt.show()

Total unique ingredients: 14,883

Top 25 ingredients:
  salt: 85,090
  butter: 54,741
  sugar: 43,974
  onion: 38,817
  water: 34,461
  eggs: 33,552
  olive oil: 32,586
  flour: 26,134
  milk: 25,672
  garlic cloves: 25,475
  pepper: 22,200
  brown sugar: 18,453
  garlic: 17,878
  all-purpose flour: 17,557
  baking powder: 17,439
  egg: 17,244
  salt and pepper: 15,335
  parmesan cheese: 14,764
  lemon juice: 14,088
  baking soda: 14,036
  vegetable oil: 13,795
  vanilla: 13,230
  black pepper: 12,955
  cinnamon: 12,485
  tomatoes: 11,896


C:\Users\Kush Shah\AppData\Local\Temp\ipykernel_16480\3137750904.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 14. Key Observations

1. **Extreme sparsity:** The user-recipe interaction matrix is >99.99% sparse, confirming the need for latent factor models (matrix factorization, neural CF) over direct nearest-neighbor approaches.

2. **Rating skew:** Ratings are heavily skewed toward 5 (positive). The implicit feedback derivation (rating >= 4 -> liked) captures the majority of interactions as positive signals.

3. **Power-law distributions:** Both user activity and recipe popularity follow power-law patterns. Most users have few interactions; most recipes receive few ratings. This creates a strong cold-start challenge.

4. **Cooking time outliers:** A small number of recipes have extreme cooking times (100,000+ minutes). These must be filtered before model training.

5. **Nutrition outliers:** Certain recipes have extreme PDV values (>1000% for some nutrients), indicating either data entry errors or recipes intended for large batches. Normalization and outlier handling are required.

6. **Temporal density:** Most data is concentrated between 2000-2018. Sequential recommendation models must account for varying temporal density across this range.

7. **Ingredient vocabulary:** Salt, butter, sugar, onions, and garlic are the most common ingredients. This high-frequency vocabulary will dominate TF-IDF representations and may need to be down-weighted.

**Next:** `03_RECIPENLG_EDA.ipynb` (Week 2) for cross-dataset comparison and NER preparation.